# Polynomial Regression: From First Principles

## 1. Why Do We Need a Curved Model?

Imagine measuring plant growth at different amounts of sunlight. Extra sunlight may help at first, but the benefit may weaken or reverse at high exposure. A straight line assumes that every additional unit of sunlight changes the predicted growth by the same amount. That constant slope may miss a systematic bend in the observations.

Polynomial Regression keeps the familiar linear regression fitting method and supplies additional features, such as $x^2$ and $x^3$. The model combines these powers to represent a curve. The extra flexibility is useful only if it captures a pattern that persists beyond the training sample.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>Build curved ingredients from the original inputs, then learn a weighted sum of those ingredients. Polynomial Regression changes the feature representation, not the basic least-squares learning principle.</div>

We will build a quadratic from a straight line, fit one by hand, extend it to multiple inputs, and connect degree selection, scaling, and regularization to the code companion. The sunlight example motivates curvature; the numerical examples below use a generic input so their values are not mistaken for biological measurements.

## 2. Inputs, Outputs, and Vocabulary

Polynomial Regression is a **supervised regression** approach. Training rows contain input features and known numerical targets. For a new input, the fitted model returns a numerical prediction. It does not directly estimate a class label or a probability.

For one input, a degree-$d$ model is

$$
\hat y_i=b+w_1x_i+w_2x_i^2+\cdots+w_dx_i^d.
$$

| Symbol or term | Meaning |
|---|---|
| $x_i$ | Original input for row $i$ |
| $y_i$, $\hat y_i$ | Observed target and prediction |
| $b$ | Intercept, the prediction at $x=0$ |
| $w_k$ | Learned coefficient of the power $x^k$ |
| $d$ | Largest power allowed in the feature expansion |
| Residual $e_i=y_i-\hat y_i$ | Observed target minus prediction |
| Polynomial feature | A power or product constructed from original inputs |
| Basis | The collection of functions whose weighted sum represents the prediction |

Degree is a **hyperparameter**, chosen using validation. Coefficients and intercept are **parameters**, estimated from training data. Specifying degree 3 permits cubic behavior; if the cubic coefficient is zero, the fitted polynomial has a lower actual degree.

Degree 1 gives a straight line, degree 2 allows a quadratic, and degree 3 allows a cubic. A nonconstant one-dimensional polynomial of degree $d$ has at most $d-1$ turning points. This is an upper bound, not a promise that every fitted curve bends that many times.

## 3. Nonlinear in the Input, Linear in the Coefficients

A straight-line model is $\hat y=b+w_1x$. Adding a squared feature gives

$$
\hat y=b+w_1x+w_2x^2.
$$

Define new features $z_1=x$ and $z_2=x^2$. Then the same equation becomes

$$
\hat y=b+w_1z_1+w_2z_2.
$$

The coefficients multiply known feature values and are added together. They are not themselves squared, multiplied together, or placed inside an unknown exponent. This is what makes the model **linear in its parameters**, even though predictions trace a curve against the original $x$.

For example, at $x=3$, the transformed row is $(3,9)$. Once the model has learned its coefficients, prediction is $b+3w_1+9w_2$. The transformer creates candidate shapes; it does not learn their usefulness from the target. The regression estimator learns their weights.

This distinction lets us use ordinary least squares, Ridge, or another linear estimator on polynomial features. Polynomial Regression is therefore a representation-plus-estimator approach, rather than one unique optimization algorithm.

## 4. Building the Design Matrix

Suppose the training inputs are $-1,0,1$. A degree-2 expansion with a constant column is

$$
\Phi=\begin{bmatrix}
1&-1&1\\
1&0&0\\
1&1&1
\end{bmatrix}.
$$

Each row represents an observation. The columns are $1$, $x$, and $x^2$. This is the **design matrix**: the numerical table used to fit the coefficients. For $n$ rows, its one-input degree-$d$ version has $n$ rows and $d+1$ columns when the constant is included.

Let $\boldsymbol\beta=(b,w_1,w_2)^T$, where $T$ means transpose. Predictions for all rows can be written compactly as $\hat{\mathbf y}=\Phi\boldsymbol\beta$. The vector $\mathbf y$ contains the observed targets.

There are two equivalent ways to represent the intercept: include a column of ones in the design matrix, or let the estimator fit it separately. The companion uses `PolynomialFeatures(include_bias=False)` and an estimator that fits an intercept. This avoids supplying a duplicate constant column.

The [PolynomialFeatures documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) describes the transformer's degree, interaction, and bias-column options. Its `fit` establishes the input schema and generated terms; it does not fit response coefficients or estimate scaling statistics.

## 5. How the Coefficients Are Learned

Ordinary least squares chooses coefficients to minimize the sum of squared residuals:

$$
J(\boldsymbol\beta)=\sum_{i=1}^{n}\left(y_i-b-\sum_{k=1}^{d}w_kx_i^k\right)^2.
$$

Squaring prevents positive and negative errors from canceling and gives larger residuals more influence. Minimizing MSE, which divides this expression by $n$, gives the same unregularized solution.

In matrix notation, the least-squares optimality equations are

$$
\Phi^T\Phi\boldsymbol\beta=\Phi^T\mathbf y.
$$

When the columns are linearly independent, these equations determine a unique coefficient vector. They are sometimes written using an inverse, but numerical solvers typically use matrix decompositions to avoid explicitly calculating it. Repeated inputs, redundant columns, or too many terms can make the coefficients nonunique or difficult to estimate accurately.

The practical sequence is simple: choose the terms, transform training inputs, fit coefficients, transform new inputs in exactly the same way, and combine them with the fitted coefficients. With noisy observations, the fitted curve usually cannot pass through every point, so it balances their squared residuals.

## 6. Worked Example: Recover a Quadratic by Hand

Retain the original notes' curve, $y=2+1.5x-0.8x^2$, but now show how its coefficients can be recovered from observations. Use three deliberately noiseless values:

| Input $x$ | Squared feature $x^2$ | Observed target $y$ |
|---|---:|---:|
| -1 | 1 | -0.3 |
| 0 | 0 | 2.0 |
| 1 | 1 | 2.7 |

Fitting $\hat y=b+w_1x+w_2x^2$ exactly requires

$$
b-w_1+w_2=-0.3,\qquad b=2,\qquad b+w_1+w_2=2.7.
$$

Insert $b=2$ into the first and third equations:

$$
-w_1+w_2=-2.3,\qquad w_1+w_2=0.7.
$$

Adding gives $2w_2=-1.6$, so $w_2=-0.8$. Substitution gives $w_1=1.5$. The resulting fitted rule is

$$
\hat y=2+1.5x-0.8x^2.
$$

All residuals are zero, so SSE is zero, the smallest possible squared error. Three distinct input values uniquely determine a polynomial of degree at most 2 that interpolates them. This is a teaching construction; with more noisy observations, ordinary least squares generally finds an approximate fit instead.

At $x=0.5$, the transformed row is $(0.5,0.25)$ and the prediction is $2+0.75-0.2=2.55$. At $x=2$, it is $2+3-3.2=1.8$. The first input is within the example's training range; the second is outside it, so the second prediction is an extrapolation.

## 7. Compare the Best Straight Line on the Same Rows

A straight line cannot pass through all three preceding points. We can calculate its least-squares fit to see what the squared term contributes.

The means are $\bar x=0$ and $\bar y=(-0.3+2+2.7)/3=22/15$. The straight-line slope is

$$
w=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2}
=\frac{3}{2}=1.5.
$$

Its intercept is $\bar y-w\bar x=22/15\approx1.4667$. Using exact fractions keeps rounding from changing the comparison:

| Input | Actual target | Line prediction | Line residual |
|---|---:|---:|---:|
| -1 | -0.3 | $-1/30$ | $-4/15$ |
| 0 | 2 | $22/15$ | $8/15$ |
| 1 | 2.7 | $89/30$ | $-4/15$ |

The line's SSE is

$$
(-4/15)^2+(8/15)^2+(-4/15)^2=96/225=32/75\approx0.4267.
$$

Negative residuals at the ends and a positive one in the middle reveal curvature the line cannot represent. The quadratic eliminates that pattern in this tiny example.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Worked-example takeaway</strong><br>The quadratic fits these training rows better because its feature set can express their bend. Exact interpolation of three points does not establish that degree 2 is best for future observations.</div>

## 8. Interpret the Curve, Not Just Individual Coefficients

In a straight line, $w_1$ is the same slope everywhere. In a quadratic, the local slope depends on the input:

$$
\frac{d\hat y}{dx}=w_1+2w_2x.
$$

A derivative describes the approximate change in prediction for a small change in input at a particular location. In our example the slope is $1.5-1.6x$: positive at zero, zero at $x=0.9375$, and negative beyond that point.

Because $w_2=-0.8<0$, the curve bends downward. Its maximum occurs at

$$
x_*=-\frac{w_1}{2w_2}=\frac{1.5}{1.6}=0.9375,
\qquad \hat y(x_*)=2.703125.
$$

Here $x_*$ denotes the turning-point input. For a fitted model, interpret such a point only if the observed range and uncertainty support it. A mathematically estimated peak is not automatically a recommended intervention or a causal optimum.

It is misleading to interpret the coefficient on $x$ while pretending $x^2$ stays fixed as $x$ changes: both features arise from the same input. Plot the entire fitted relationship or calculate changes in the complete prediction equation. Centering and scaling change individual coefficients too, even when equivalent predictions are possible.

## 9. Multiple Inputs and Interactions

With two original inputs, a full degree-2 model includes

$$
\hat y=b+w_1x_1+w_2x_2+w_{11}x_1^2+w_{12}x_1x_2+w_{22}x_2^2.
$$

The cross-product $x_1x_2$ is an **interaction**. It lets the effect of one input depend on the other. For example, fertilizer response may differ with sunlight, so their joint contribution may not equal two independent effects.

For a row $(x_1,x_2)=(2,3)$, the nonconstant features are $(2,3,4,6,9)$. The total degree of $x_1x_2$ is 2 because its exponents sum to 2. A term $x_1^2x_2$ has total degree 3 and is excluded from a full degree-2 expansion.

If the interaction coefficient $w_{12}$ is nonzero, the local slope for $x_1$ is

$$
\frac{\partial\hat y}{\partial x_1}=w_1+2w_{11}x_1+w_{12}x_2.
$$

The partial derivative describes changes in $x_1$ while keeping the other original input $x_2$ fixed. Its dependence on $x_2$ makes the interaction explicit. This remains a predictive association unless the data and study design support a causal interpretation.

With `interaction_only=True`, scikit-learn generates products of distinct inputs and omits repeated powers such as $x_1^2$. For two features at degree 2, the nonconstant terms are $x_1,x_2,x_1x_2$. For a single feature, this setting does not create a quadratic or cubic curve merely by increasing degree.

## 10. How Quickly the Feature Set Grows

For $p$ original features and all terms of total degree at most $d$, the number of columns including the constant is

$$
\binom{p+d}{d}=\frac{(p+d)!}{p!d!}.
$$

The exclamation mark denotes factorial: $k!$ is the product of integers from 1 through $k$, with $0!=1$. Excluding the constant subtracts one from the count. This formula is for full polynomial expansion, not interaction-only expansion.

| Original features $p$ | Maximum degree $d$ | Nonconstant expanded features |
|---|---:|---:|
| 1 | 2 | 2 |
| 2 | 2 | 5 |
| 10 | 2 | 65 |
| 10 | 3 | 285 |
| 20 | 3 | 1770 |

Let $q$ be the number of expanded nonconstant features. Storing a dense expanded dataset takes roughly $O(nq)$ memory. Dense linear prediction uses roughly $O(q)$ coefficient arithmetic per row after generating the features. Training cost depends on the solver; a direct tall-matrix least-squares approach can require roughly $O(nq^2)$ work when $n$ is at least as large as $q$.

Thus, a modest-looking degree can be expensive when many inputs are present. It also asks the data to support many coefficients. Include transformations that make sense for the problem instead of expanding every column indiscriminately.

## 11. Degree, Bias, Variance, and Interpolation

A degree that is too low may miss a real bend. This is **underfitting**. A very flexible polynomial may chase sample noise, producing unstable predictions between observations or near boundaries. This is **overfitting**.

**Bias** is systematic error in the average fitted prediction over repeated training samples. **Variance** describes how much those predictions change across samples. Increasing degree can reduce approximation bias while increasing variance; neither training fit nor visual smoothness alone resolves the tradeoff.

For exact unregularized least squares on the same rows with nested feature sets, adding terms cannot increase the minimum training SSE. The larger model can always set its new coefficients to zero. This is why training error alone cannot select degree. Numerical errors, changing preprocessing, or regularized comparisons can complicate observed results.

A polynomial of degree at most $n-1$ can interpolate $n$ distinct one-dimensional inputs exactly, but that fact says nothing about prediction between them or beyond them. Repeated inputs with conflicting targets cannot all be fitted exactly by any deterministic function of that input.

Choose degree using training-only validation and compare with degree 1. A small candidate range can be a useful starting point, but no degree is universally correct. If several settings perform similarly within validation uncertainty, consider their stability, complexity, and practical interpretability.

## 12. Scaling, Numerical Stability, and Regularization

Powers can grow quickly: when $x=10$, $x^8=100{,}000{,}000$. Columns of very different magnitudes and highly correlated powers can make coefficient estimation numerically sensitive. A small coefficient on a high power may still produce a large contribution, so raw coefficient size is not a reliable importance ranking.

Centering or scaling original inputs before expansion can keep powers better behaved. Scaling expanded columns can make a common regularization penalty more meaningful. These operations have different roles: scaling $x$ before squaring is not the same transformation as standardizing $x^2$ afterward.

With a complete polynomial basis, an invertible shift or rescaling of the original inputs can represent the same class of unregularized functions in exact arithmetic. Regularization is not invariant to these changes: it penalizes coordinates in the chosen representation. Document the order and validate the entire pipeline.

Ridge on expanded features minimizes, in the scikit-learn convention,

$$
J=\sum_i\left(y_i-b-\sum_{j=1}^{q}w_jz_{ij}\right)^2+\alpha\sum_{j=1}^{q}w_j^2,
$$

where $z_{ij}$ is expanded feature $j$ for row $i$, $q$ is the expanded feature count, and $\alpha\geq0$ controls the penalty. The intercept is not penalized. Ridge can reduce unstable coefficient combinations, but it does not directly impose a bound on curvature everywhere.

Lasso and Elastic Net can also select among polynomial columns. Without extra constraints, they may keep a squared or interaction term while removing its lower-order terms. Decide whether that structure makes sense for the task.

<div style="color:#22543d; background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>Practical tip</strong><br>A useful regularized workflow is polynomial expansion, training-fitted scaling of expanded columns, then Ridge. For large raw magnitudes, consider scaling before expansion as well. Select degree and penalty together within cross-validation.</div>

## 13. Training, Validation, and Prediction Workflow

1. Define the target and available prediction-time features. Inspect units, missing values, outliers, and the range in which predictions will be used.
2. Reserve a final test set before selecting degree or penalty. Use time-aware or group-aware splits when observations are dependent.
3. Build a pipeline containing any imputation, the chosen transformations, any scaling, and the regression estimator.
4. Cross-validate candidate degrees and regularization strengths on development training data. Fit every learned preprocessing step on each fold's training rows only.
5. Compare scores and residual patterns with a degree-1 baseline. Select the configuration using validation, then refit it on all development training rows.
6. Evaluate once on the reserved test set and apply the same fitted pipeline to new raw inputs.

For a pipeline using the companion's names, `poly__degree` controls expansion and `model__alpha` controls Ridge strength. Compare both settings jointly when using Ridge. Do not tune repeatedly against the final test score.

PolynomialFeatures itself creates fixed powers rather than estimating moments from observed values. Nonetheless, keeping it in the pipeline prevents inconsistent transformations and ensures degree choices participate in the same validation procedure. Scalers, imputers, and feature selection do learn from data and must stay inside each training fold.

Do not blindly raise category codes to powers: their numeric distances may be meaningless. One-hot indicators can also create redundant powers, since squaring a zero-or-one indicator returns itself. Build representations appropriate to the feature types.

## 14. Connecting to the Code Companion

`06_Polynomial_Regression.ipynb` generates 120 equally spaced inputs between -3 and 3, then adds random noise to the signal

$$
y=2+1.5x-0.8x^2+\varepsilon.
$$

Here $\varepsilon$ is noise generated from a normal distribution with mean zero and standard deviation 1. The code uses random seed 42 and reserves 25% of rows as a test set. The known signal is quadratic by construction; its coefficients are not claimed to be the coefficients estimated from the noisy training rows.

| Companion model | Representation and estimator |
|---|---|
| Linear | Original input with `LinearRegression` |
| Polynomial degree 2 | `PolynomialFeatures(degree=2, include_bias=False)` with `LinearRegression` |
| Polynomial degree 8 with Ridge | Powers through degree 8 with `Ridge(alpha=1.0)` |

The Ridge pipeline does not scale its expanded columns. On the original input range, an eighth power can reach $3^8=6561$, while the first power has magnitude at most 3. Its penalty therefore depends strongly on the unscaled basis. It is an illustration, not a fully tuned degree-and-penalty comparison. The degree-2 and degree-8 models also differ in both degree and regularization, so their score difference cannot isolate either choice alone.

The plotting grid extends from -3.2 to 3.2. Values outside [-3,3] are outside the entire generated input range, and the training subset can have an even narrower range. A plotted curve is a model output, not additional evidence from observed targets.

These notes describe the companion's code and known generating formula. They do not report measured scores, fitted coefficients, or a winning model because the companion was not executed for this revision.

## 15. Evaluate Error and Diagnose Remaining Structure

For $m$ held-out observations with residuals $e_i=y_i-\hat y_i$:

$$
\mathrm{MAE}=\frac1m\sum_i|e_i|,\qquad
\mathrm{MSE}=\frac1m\sum_i e_i^2,\qquad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}.
$$

MAE is average absolute error in target units. RMSE also has target units but gives large errors more influence. Lower is better on the same evaluation rows. Regularization penalties are not added to held-out prediction metrics.

For nonconstant evaluation targets,

$$
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y_{\mathrm{eval}})^2},
$$

where $\bar y_{\mathrm{eval}}$ is their mean. One is perfect, zero matches the evaluation-mean reference in this formula, and negative values are worse. The ratio is undefined for constant targets. Compare against a deployable baseline fitted to the training target mean as well.

For one input, plot observations, fitted curves, and residuals across the input range. A remaining curve in residuals suggests an inadequate representation; increasing residual spread suggests unequal error variability. Large isolated residuals may indicate outliers or missing features. These are diagnostic clues, not automatic reasons to raise degree.

When drawing curves, use a sorted input grid so plotting connects points in input order. For multiple inputs, inspect predictions while holding other inputs at plausible values or use clearly defined summaries. A single two-dimensional curve cannot reveal every interaction.

## 16. Interpolation, Extrapolation, and Alternatives

**Interpolation** predicts within the region supported by training inputs. **Extrapolation** predicts outside it. A polynomial is defined everywhere, but its behavior outside the observed region is controlled by assumptions that have not been checked there.

For our example, the quadratic predicts $2+15-80=-63$ at $x=10$. That arithmetic is correct, but if the target represents a nonnegative quantity it could be physically impossible. Inside a limited interval a polynomial can approximate saturation; it does not generally remain flat as inputs grow without bound.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Common pitfall</strong><br>A smooth curve and a good random-split score do not validate predictions far beyond the training range. Regularization can reduce instability without making extrapolation reliable.</div>

With multiple inputs, a new combination may be poorly supported even if each individual value lies within its observed minimum and maximum. Validation should resemble the combinations and range expected in deployment.

| Alternative | Why compare it |
|---|---|
| Straight-line model | The extra curvature may not improve validated predictions |
| Low-degree polynomial with Ridge | A compact smooth curve with controlled coefficients may suffice |
| Splines | Piecewise polynomial functions can offer more local flexibility |
| Trees or forests | Split-based models can represent interactions without enumerating all powers |
| A domain-specific functional form | Known saturation, positivity, or physical constraints may be important |

Each alternative has its own assumptions and tuning needs. Splines and tree models do not automatically solve extrapolation, and polynomial coefficients do not establish causality.

## 17. Assumptions, Limitations, and Recap

Polynomial regression is useful when a smooth curved relationship is plausible over the relevant input range and the feature expansion remains manageable. It does not require normally distributed errors to compute a least-squares fit, although stronger assumptions are needed for particular inferential claims. Useful validation requires representative data and appropriate handling of dependent rows.

| Failure mode | Why it occurs | Practical response |
|---|---|---|
| Underfitting | Selected terms cannot describe the pattern | Evaluate a better representation or another model |
| Overfitting | Many coefficients follow sample noise | Validate degree and regularization |
| Numerical instability | Powers differ greatly in scale or overlap strongly | Scale thoughtfully and reduce unnecessary terms |
| Large feature expansion | Powers and interactions multiply rapidly | Check feature counts and select meaningful terms |
| Outlier sensitivity | Squared loss gives large residuals strong influence | Inspect data quality and consider robust fitting |
| Implausible extrapolation | Polynomial behavior outside the observed range is unconstrained | Restrict use or adopt a justified domain model |

The central idea is to transform inputs into powers and products, then learn a linear combination of those transformed features. Degree controls the available shapes, while regularization controls how their coefficients are used. These choices should be assessed together using an independent validation design.

Remember the worked example: fitting a quadratic exactly showed how coefficients are learned, not that exact training fit is the objective of model selection. Prefer a curve whose predictions are supported by data, stable under validation, and plausible over the range where it will be used.